# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

## OCR Quality — CER / WER against the gold standard
Evaluates Stage 3's actual OCR accuracy against `grading_kit/labels.jsonl` (the independently
human-reviewed page sample) — a continuous accuracy measurement, distinct from the pass/fail
CER gate in `vision/ocr.py` (which only tags a line `"gold"` on an exact CER==0.0 match). A
line that fails that strict gate is still scored here, so this reports the real OCR error
rate, not just the gate's pass rate.

CER = character-level edit distance / reference length. WER is the same idea at word
granularity: WER = word-level edit distance / reference word count.

In [12]:
import sys
from pathlib import Path

# Find the project root and add the 'src' directory to sys.path
current = Path.cwd().resolve()
while current != current.parent:
    src_dir = current / "src"
    if src_dir.exists() and (src_dir / "doc_agent").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
    current = current.parent

import json
from doc_agent.ingest import loader, preprocess
from doc_agent.vision import layout, ocr
from doc_agent.vision.ocr import _normalize, _levenshtein

In [13]:
import json
import sys
from pathlib import Path

try:
    import yaml
except ImportError:
    !pip install pyyaml
    import yaml

# Find project root directory dynamically (searches upwards for configs or src)
current = Path.cwd().resolve()
project_root = current
while current != current.parent:
    if (current / "configs").exists() or (current / "src").exists():
        project_root = current
        break
    current = current.parent

# Load configuration relative to project root
config_path = project_root / "configs" / "config.yaml"

if config_path.exists():
    with open(config_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    print(f"Loaded configuration from: {config_path}")
else:
    cfg = {}
    print(f"Warning: Config file not found at {config_path}")

# Load gold standard labels relative to project root
labels_rel = cfg.get("grading_kit", {}).get("labels_path", "grading_kit/labels.jsonl")
labels_path = Path(labels_rel)
if not labels_path.is_absolute():
    labels_path = project_root / labels_path

gold_texts: dict[str, str] = {}
if labels_path.exists():
    with open(labels_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            row = json.loads(line)
            pid, text = row.get("page_id"), row.get("text", "")
            if pid and text and not text.startswith("REPLACE ME"):
                gold_texts[pid] = text

print(f"Loaded {len(gold_texts)} gold-labelled pages from {labels_path}")
if not gold_texts:
    print(
        "No gold labels yet -- fill grading_kit/labels.jsonl with reviewed page "
        "transcriptions before this section can report anything."
    )

Loaded configuration from: /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/configs/config.yaml
Loaded 5 gold-labelled pages from /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/grading_kit/labels.jsonl


In [14]:
# 1. Ensure configuration paths use resolved absolute paths
cfg.setdefault("paths", {})
cfg["paths"]["raw_dir"] = str((project_root / "data" / "raw").resolve())
cfg["paths"]["processed_dir"] = str(
    (project_root / "data" / "processed").resolve()
)

# 2. Guarantee the processed output directory exists
processed_dir = Path(cfg["paths"]["processed_dir"]).resolve()
processed_dir.mkdir(parents=True, exist_ok=True)

try:
    # 3. Run Stages 1-3 only for gold-labelled pages
    raw_pages = [p for p in loader.load_pages(cfg) if p.id in gold_texts]
    print(f"Raw pages: {len(raw_pages)}")
    pages = preprocess.run(raw_pages, cfg)
    regions = layout.detect(pages, cfg)
    ocr.transcribe(
        regions, cfg
    )  # writes data/processed/ocr_meta.jsonl + layout_meta.jsonl

    # 4. Parse OCR outputs safely using a context manager
    ocr_file = processed_dir / "ocr_meta.jsonl"
    ocr_rows = []
    if ocr_file.exists():
        with open(ocr_file, encoding="utf-8") as f:
            ocr_rows = [json.loads(line) for line in f if line.strip()]

    by_page: dict[str, list[str]] = {}
    for row in sorted(ocr_rows, key=lambda r: r["region_id"]):
        by_page.setdefault(row["page_id"], []).append(
            row["ocr_text_normalized"]
        )

    print(f"OCR'd {len(pages)} gold-labelled pages, {len(ocr_rows)} lines total")

except FileNotFoundError as e:
    by_page = {}
    print("Skipping -- corpus not available yet in this environment:")
    print(" ", e)

{"ts":"2026-08-10 18:53:57,791","lvl":"INFO","mod":"doc_agent.ingest.loader","msg":"loaded 833 pages across 6 documents from /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/raw"}
Raw pages: 5
{"ts":"2026-08-10 18:54:03,489","lvl":"INFO","mod":"doc_agent.ingest.preprocess","msg":"preprocessed 5 pages, dropped 0 blank/separator pages"}
{"ts":"2026-08-10 18:54:06,888","lvl":"INFO","mod":"doc_agent.vision.layout","msg":"detected 81 line regions across 5 pages -> /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/processed/layout_meta.jsonl"}
{"ts":"2026-08-10 18:54:11,272","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"OCR'd 81 regions across 5 pages: 0 accepted -> 0 chunks, 81 rejected -> /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/processed/ocr_meta.jsonl"}
OCR'd 5 gold-labelled pages, 81 lines total


In [15]:
# 3. Score CER/WER: keep the line-level gate for layout validation, but also compute a page-level
# score so the notebook still reports meaningful OCR quality when the detected line count does not
# exactly match the gold page transcription.
def _wer(hyp_words: list[str], ref_words: list[str]) -> float:
    """Word Error Rate = word-level edit distance / reference word count."""
    if not ref_words:
        raise ValueError('_wer() requires a non-empty reference')
    return _levenshtein(hyp_words, ref_words) / len(ref_words)

line_results = []
page_results = []
mismatched_pages = []
for page_id, gold_text in gold_texts.items():
    hyp_lines = by_page.get(page_id)
    if hyp_lines is None:
        continue
    ref_lines = [ln.strip() for ln in gold_text.splitlines() if ln.strip()]
    if not ref_lines:
        continue

    hyp_page_text = _normalize("\n".join(hyp_lines))
    ref_page_text = _normalize("\n".join(ref_lines))
    page_cer = _levenshtein(hyp_page_text, ref_page_text) / len(ref_page_text)
    page_wer = _wer(hyp_page_text.split(), ref_page_text.split()) if ref_page_text.split() else None
    page_results.append({
        'page_id': page_id,
        'ocr_lines': len(hyp_lines),
        'gold_lines': len(ref_lines),
        'page_cer': page_cer,
        'page_wer': page_wer,
        'hyp_page_text': hyp_page_text,
        'ref_page_text': ref_page_text,
    })

    if len(hyp_lines) != len(ref_lines):
        mismatched_pages.append((page_id, len(hyp_lines), len(ref_lines)))
        continue

    for hyp, ref in zip(hyp_lines, ref_lines):
        ref_norm = _normalize(ref)
        if not ref_norm:
            continue
        cer = _levenshtein(hyp, ref_norm) / len(ref_norm)
        hyp_words, ref_words = hyp.split(), ref_norm.split()
        wer = _wer(hyp_words, ref_words) if ref_words else None
        line_results.append({
            'page_id': page_id,
            'cer': cer,
            'wer': wer,
            'hyp': hyp,
            'ref': ref_norm,
        })

print(f'scored {len(line_results)} lines across {len({r["page_id"] for r in line_results})} gold pages')
print(f'scored {len(page_results)} gold pages at page level')
if mismatched_pages:
    print(f'skipped line-level scoring for {len(mismatched_pages)} page(s) with mismatched line counts '
          f'(ocr lines vs. gold lines): {mismatched_pages}')

scored 0 lines across 0 gold pages
scored 5 gold pages at page level
skipped line-level scoring for 5 page(s) with mismatched line counts (ocr lines vs. gold lines): [('arogya_p0053', 26, 31), ('bishwaparichay_p0343', 31, 6), ('chandalika_p0161', 7, 27), ('chitrangada_p0130', 11, 31), ('tin_sangi_p0219', 6, 11)]


In [16]:
# 4. Display aggregate CER / WER
import pandas as pd

if line_results:
    mean_cer = sum(r['cer'] for r in line_results) / len(line_results)
    wer_values = [r['wer'] for r in line_results if r['wer'] is not None]
    mean_wer = sum(wer_values) / len(wer_values) if wer_values else float('nan')
    print(f'Line-level mean CER: {mean_cer:.4f}')
    print(f'Line-level mean WER: {mean_wer:.4f}')
    df = pd.DataFrame(line_results)[['page_id', 'cer', 'wer', 'hyp', 'ref']]
    df
else:
    print('No pages had exact line-count matches, so line-level scoring was skipped.')

if page_results:
    mean_page_cer = sum(r['page_cer'] for r in page_results) / len(page_results)
    page_wer_values = [r['page_wer'] for r in page_results if r['page_wer'] is not None]
    mean_page_wer = sum(page_wer_values) / len(page_wer_values) if page_wer_values else float('nan')
    print(f'Page-level mean CER: {mean_page_cer:.4f}')
    print(f'Page-level mean WER: {mean_page_wer:.4f}')
    page_df = pd.DataFrame(page_results)[['page_id', 'ocr_lines', 'gold_lines', 'page_cer', 'page_wer']]
    page_df
else:
    print('Nothing scored -- check that grading_kit/labels.jsonl has real entries and that '
          'the corresponding pages exist under data/raw/.')

No pages had exact line-count matches, so line-level scoring was skipped.
Page-level mean CER: 0.6515
Page-level mean WER: 0.7817
